# Combine Normal EEG Features Into One CSV

This notebook builds one combined dataset from:
- resting/thinking MAV and variance CSV files
- ERD/ERS band power CSV
- encoding chaining CSV

It keeps only normal subjects with IDs like `id1` or `id_1` and excludes ALS rows.

## 1) Configure Notebook Paths and Imports

Import required packages and set paths for input/output files.

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

ROOT = Path('.').resolve()
OUTPUT_CSV = ROOT / 'normal_features_combined.csv'

print(f'Working directory: {ROOT}')
print(f'Output CSV path: {OUTPUT_CSV}')

Working directory: /home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_ALS/WEB/features.csv
Output CSV path: /home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_ALS/WEB/features.csv/normal_features_combined.csv


## 2) Define Subject Filter and Metadata Parsers

Create helpers to detect normal IDs and parse task/feature/channel/subband metadata.

In [6]:
NORMAL_SUBJECT_PATTERN = re.compile(r'^id_?\d+$', flags=re.IGNORECASE)
CHANNELS_KEEP = {'C3', 'C4', 'CZ'}
SUBBANDS_KEEP = {'High_Beta', 'Low_Beta', 'Mu'}
KEY_COLS = ['subject_id', 'scenario', 'task', 'channel', 'subband']


def normalize_subject(value):
    if pd.isna(value):
        return np.nan
    return str(value).strip().lower()


def is_normal_subject(value):
    if pd.isna(value):
        return False
    v = str(value).strip().lower()
    return bool(NORMAL_SUBJECT_PATTERN.match(v)) and ('als' not in v)


def normalize_scenario(value):
    if pd.isna(value):
        return np.nan
    return str(value).strip().lower()


def normalize_task(value):
    if pd.isna(value):
        return np.nan
    return str(value).strip().title()


def normalize_channel(value):
    if pd.isna(value):
        return np.nan
    return str(value).strip().upper().replace(' ', '')


def normalize_subband(value):
    if pd.isna(value):
        return np.nan
    raw = str(value).strip().replace(' ', '_').replace('-', '_').lower()
    mapping = {
        'high_beta': 'High_Beta',
        'low_beta': 'Low_Beta',
        'mu': 'Mu',
    }
    return mapping.get(raw, raw)


def parse_task_feature_from_filename(file_path):
    # Accept mixed case and optional spaces around "-" in exported filenames.
    pattern = r'^(resting|thinking)_(mav|var)_normal\s*-\s*(c3|c4|cz)\.csv$'
    match = re.match(pattern, file_path.name, flags=re.IGNORECASE)
    if match is None:
        return None

    task_raw, feature_raw, channel_raw = match.groups()
    return {
        'task': task_raw.title(),
        'feature_type': 'mav' if feature_raw.lower() == 'mav' else 'variance',
        'channel': channel_raw.upper(),
    }

## 3) Collect CSV Files for Task Features, ERD/ERS, and Encoding

Scan the folder and build an index of files plus parsed metadata.

In [7]:
task_feature_files = []
for path in ROOT.rglob('*.csv'):
    meta = parse_task_feature_from_filename(path)
    if meta is not None:
        task_feature_files.append({'file_path': path, **meta})

task_feature_index = pd.DataFrame(task_feature_files)
if not task_feature_index.empty:
    task_feature_index = task_feature_index.sort_values(
        ['task', 'feature_type', 'channel']
    ).reset_index(drop=True)

erd_files = sorted(ROOT.rglob('*ERD_ERS*.csv'))
encoding_files = sorted([p for p in ROOT.rglob('*.csv') if p.name.lower() == 'encoding chaining.csv'])

print(f'Task-feature files found: {len(task_feature_index)}')
if not task_feature_index.empty:
    display(task_feature_index)

print(f'ERD/ERS files found: {[p.name for p in erd_files]}')
print(f'Encoding files found: {[p.name for p in encoding_files]}')

Task-feature files found: 12


,file_path,task,feature_type,channel
0,/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_AL...,Resting,mav,C3
1,/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_AL...,Resting,mav,C4
2,/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_AL...,Resting,mav,CZ
3,/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_AL...,Resting,variance,C3
4,/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_AL...,Resting,variance,C4
5,/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_AL...,Resting,variance,CZ
6,/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_AL...,Thinking,mav,C3
7,/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_AL...,Thinking,mav,C4
8,/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_AL...,Thinking,mav,CZ
9,/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_AL...,Thinking,variance,C3


ERD/ERS files found: ['erd_ers_band_power.xlsx - ERD_ERS.csv']
Encoding files found: ['encoding chaining.csv']


## 4) Load and Normalize Task Feature CSVs (resting, thinking_var, mav_normal)

Read each feature CSV and reshape each subband block (High_Beta, Low_Beta, Mu) into tidy records.

In [8]:
def load_task_feature_csv(file_path, task, feature_type, channel):
    raw = pd.read_csv(file_path, header=1)
    raw = raw.iloc[:, 1:]

    if raw.shape[1] < 30:
        raise ValueError(
            f'{file_path.name} has {raw.shape[1]} columns after cleanup. Expected at least 30.'
        )

    scenarios = [f'scenario{i}' for i in range(1, 10)]
    subbands = ['High_Beta', 'Low_Beta', 'Mu']
    pieces = []

    for block_idx, subband in enumerate(subbands):
        block = raw.iloc[:, block_idx * 10:(block_idx + 1) * 10].copy()
        block.columns = ['subject_id'] + scenarios
        block['subject_id'] = block['subject_id'].map(normalize_subject)
        block = block[block['subject_id'].map(is_normal_subject)].copy()

        long_block = block.melt(
            id_vars='subject_id',
            var_name='scenario',
            value_name='value'
        )
        long_block['scenario'] = long_block['scenario'].map(normalize_scenario)
        long_block['task'] = task
        long_block['feature_type'] = feature_type
        long_block['channel'] = normalize_channel(channel)
        long_block['subband'] = subband
        long_block['source_file'] = file_path.name
        pieces.append(long_block)

    out = pd.concat(pieces, ignore_index=True)
    out['value'] = pd.to_numeric(out['value'], errors='coerce')
    out = out.dropna(subset=['value'])
    out.columns = [col.strip().lower() for col in out.columns]
    return out


if task_feature_index.empty:
    raise FileNotFoundError('No task feature files were found with the expected naming pattern.')

task_feature_frames = []
for _, row in task_feature_index.iterrows():
    one_df = load_task_feature_csv(
        file_path=row['file_path'],
        task=row['task'],
        feature_type=row['feature_type'],
        channel=row['channel'],
    )
    task_feature_frames.append(one_df)

task_feature_long_raw = pd.concat(task_feature_frames, ignore_index=True)
print(f'Loaded task-feature long rows: {len(task_feature_long_raw):,}')
display(task_feature_long_raw.head())

Loaded task-feature long rows: 55,062


,subject_id,scenario,value,task,feature_type,channel,subband,source_file
0,id1,scenario1,5.082302,Resting,mav,C3,High_Beta,resting_mav_normal - C3.csv
1,id10,scenario1,2.022243,Resting,mav,C3,High_Beta,resting_mav_normal - C3.csv
2,id100,scenario1,2.481057,Resting,mav,C3,High_Beta,resting_mav_normal - C3.csv
3,id101,scenario1,6.911049,Resting,mav,C3,High_Beta,resting_mav_normal - C3.csv
4,id102,scenario1,17.033939,Resting,mav,C3,High_Beta,resting_mav_normal - C3.csv


## 5) Keep Only Normal Subjects (`id_xx`) and EEG Columns (`c3`, `c4`, `cz`)

Apply subject/channel/subband filters and keep only required keys and value columns.

In [9]:
task_feature_long = task_feature_long_raw.copy()
task_feature_long['subject_id'] = task_feature_long['subject_id'].map(normalize_subject)
task_feature_long['scenario'] = task_feature_long['scenario'].map(normalize_scenario)
task_feature_long['task'] = task_feature_long['task'].map(normalize_task)
task_feature_long['channel'] = task_feature_long['channel'].map(normalize_channel)
task_feature_long['subband'] = task_feature_long['subband'].map(normalize_subband)
task_feature_long['feature_type'] = task_feature_long['feature_type'].astype(str).str.strip().str.lower()

task_feature_long = task_feature_long[
    task_feature_long['subject_id'].map(is_normal_subject)
    & task_feature_long['channel'].isin(CHANNELS_KEEP)
    & task_feature_long['subband'].isin(SUBBANDS_KEEP)
    & task_feature_long['feature_type'].isin(['mav', 'variance'])
].copy()

task_feature_long = task_feature_long[
    ['subject_id', 'scenario', 'task', 'feature_type', 'channel', 'subband', 'value', 'source_file']
].reset_index(drop=True)

print(f'Rows after normal/channel/subband filter: {len(task_feature_long):,}')
print(f'Unique subjects after filter: {task_feature_long["subject_id"].nunique():,}')
display(task_feature_long.head())

Rows after normal/channel/subband filter: 55,062
Unique subjects after filter: 170


,subject_id,scenario,task,feature_type,channel,subband,value,source_file
0,id1,scenario1,Resting,mav,C3,High_Beta,5.082302,resting_mav_normal - C3.csv
1,id10,scenario1,Resting,mav,C3,High_Beta,2.022243,resting_mav_normal - C3.csv
2,id100,scenario1,Resting,mav,C3,High_Beta,2.481057,resting_mav_normal - C3.csv
3,id101,scenario1,Resting,mav,C3,High_Beta,6.911049,resting_mav_normal - C3.csv
4,id102,scenario1,Resting,mav,C3,High_Beta,17.033939,resting_mav_normal - C3.csv


## 6) Reshape by Channel/Subband and Tag Task + Feature Type

Convert long MAV/variance rows into one consistent wide table with separate `mav` and `variance` columns.

In [10]:
main_feature_wide = (
    task_feature_long
    .pivot_table(
        index=['subject_id', 'scenario', 'task', 'channel', 'subband'],
        columns='feature_type',
        values='value',
        aggfunc='first',
    )
    .reset_index()
)
main_feature_wide.columns.name = None

print(f'Main feature rows (wide): {len(main_feature_wide):,}')
display(main_feature_wide.head())

Main feature rows (wide): 27,531


,subject_id,scenario,task,channel,subband,mav,variance
0,id1,scenario1,Resting,C3,High_Beta,5.082302,0.000089
1,id1,scenario1,Resting,C3,Low_Beta,3.395974,0.000034
2,id1,scenario1,Resting,C3,Mu,3.931533,0.000055
3,id1,scenario1,Resting,C4,High_Beta,4.151432,0.000044
4,id1,scenario1,Resting,C4,Low_Beta,2.680949,0.000018


## 7) Merge ERD/ERS Band Power and `encoding chaining.csv`

Load ERD/ERS and encoding files, normalize keys, filter to normal IDs, and merge into the main feature table.

In [14]:
def find_first_column(df, prefix):
    matches = [c for c in df.columns if str(c).lower().startswith(prefix.lower())]
    if not matches:
        raise KeyError(f'No column starts with: {prefix}')
    return matches[0]


if not erd_files:
    raise FileNotFoundError('ERD/ERS file was not found.')

erd = pd.read_csv(erd_files[0])
erd['subject_id'] = erd['subject'].map(normalize_subject)
erd['scenario'] = erd['scenario'].map(normalize_scenario)
erd['task'] = erd['task'].map(normalize_task)
erd['channel'] = erd['channel'].map(normalize_channel)
erd['subband'] = erd['subband'].map(normalize_subband)
erd['category'] = erd['category'].astype(str).str.strip().str.lower()

baseline_col = find_first_column(erd, 'baseline_power')
task_power_col = find_first_column(erd, 'task_power')

erd = erd.rename(columns={
    baseline_col: 'baseline_power',
    task_power_col: 'task_power',
})

erd = erd[
    (erd['category'] == 'normal')
    & erd['subject_id'].map(is_normal_subject)
    & erd['channel'].isin(CHANNELS_KEEP)
    & erd['subband'].isin(SUBBANDS_KEEP)
].copy()

ERD_SCALE = 10_000
for col in ['baseline_power', 'task_power', 'erd_ers_pct']:
    # Keep NaN values untouched while scaling all valid numeric values.
    erd[col] = pd.to_numeric(erd[col], errors='coerce') * ERD_SCALE

erd_use = erd[
    ['subject_id', 'scenario', 'task', 'channel', 'subband', 'baseline_power', 'task_power', 'erd_ers_pct']
].drop_duplicates()


if not encoding_files:
    raise FileNotFoundError('encoding chaining.csv was not found.')

encoding_chunks = []
for chunk in pd.read_csv(
    encoding_files[0],
    usecols=['subject_id', 'scenario', 'task', 'channel', 'subband', 'feature', 'chain_sequence', 'chain_ratio'],
    chunksize=250000,
):
    chunk['subject_id'] = chunk['subject_id'].map(normalize_subject)
    chunk['scenario'] = chunk['scenario'].map(normalize_scenario)
    chunk['task'] = chunk['task'].map(normalize_task)
    chunk['channel'] = chunk['channel'].map(normalize_channel)
    chunk['subband'] = chunk['subband'].map(normalize_subband)
    chunk['feature'] = chunk['feature'].astype(str).str.strip().str.lower()

    chunk = chunk[
        chunk['subject_id'].map(is_normal_subject)
        & chunk['channel'].isin(CHANNELS_KEEP)
        & chunk['subband'].isin(SUBBANDS_KEEP)
        & chunk['feature'].isin(['mav', 'variance'])
    ].copy()

    chunk['chain_ratio'] = pd.to_numeric(chunk['chain_ratio'], errors='coerce')
    encoding_chunks.append(chunk)

if encoding_chunks:
    encoding_long = pd.concat(encoding_chunks, ignore_index=True).drop_duplicates()
else:
    encoding_long = pd.DataFrame(
        columns=['subject_id', 'scenario', 'task', 'channel', 'subband', 'feature', 'chain_sequence', 'chain_ratio']
    )

if not encoding_long.empty:
    encoding_pivot = (
        encoding_long
        .pivot_table(
            index=['subject_id', 'scenario', 'task', 'channel', 'subband'],
            columns='feature',
            values=['chain_sequence', 'chain_ratio'],
            aggfunc='first',
        )
        .reset_index()
    )
    encoding_pivot.columns = [
        '_'.join([str(part) for part in col if str(part) != '']) if isinstance(col, tuple) else col
        for col in encoding_pivot.columns
    ]
else:
    encoding_pivot = pd.DataFrame(
        columns=[
            'subject_id', 'scenario', 'task', 'channel', 'subband',
            'chain_sequence_mav', 'chain_ratio_mav',
            'chain_sequence_variance', 'chain_ratio_variance',
        ]
    )

combined = main_feature_wide.merge(
    erd_use,
    on=['subject_id', 'scenario', 'task', 'channel', 'subband'],
    how='left',
)
combined = combined.merge(
    encoding_pivot,
    on=['subject_id', 'scenario', 'task', 'channel', 'subband'],
    how='left',
)

print(f'ERD/ERS rows after filter: {len(erd_use):,}')
print(f'Encoding rows after filter: {len(encoding_long):,}')
print(f'Combined rows after merge: {len(combined):,}')
display(combined.head())

ERD/ERS rows after filter: 13,743
Encoding rows after filter: 91,692
Combined rows after merge: 27,531


,subject_id,scenario,task,channel,subband,mav,variance,baseline_power,task_power,erd_ers_pct,chain_ratio_mav,chain_ratio_variance,chain_sequence_mav,chain_sequence_variance
0,id1,scenario1,Resting,C3,High_Beta,5.082302,0.000089,NaN,NaN,NaN,0.5488,0.5305,1001010110000111011000111001101000101001010110...,1001010110000111011000111001101000101001010110...
1,id1,scenario1,Resting,C3,Low_Beta,3.395974,0.000034,NaN,NaN,NaN,0.5610,0.5610,1001110101010111111010110101101100101101011110...,1001110111010111111010110001101100101101011110...
2,id1,scenario1,Resting,C3,Mu,3.931533,0.000055,NaN,NaN,NaN,0.5244,0.5183,1101000101011011101100010011010011001101100101...,1101000101011011101100010011010011001101100101...
3,id1,scenario1,Resting,C4,High_Beta,4.151432,0.000044,NaN,NaN,NaN,0.5671,0.5610,1010111101001101110011010111010011010010101000...,1010111101001101110011010111010011010010101000...
4,id1,scenario1,Resting,C4,Low_Beta,2.680949,0.000018,NaN,NaN,NaN,0.5305,0.5305,1011011101001101110001100011011011010000111000...,1011011101001101110001100011011011010000111000...


## 8) Run Data Quality Checks and Export a Single Combined CSV

Validate key uniqueness, null rates, channel/subband coverage, then export one final CSV.

In [15]:
combined['subject_num'] = pd.to_numeric(
    combined['subject_id'].str.extract(r'(\d+)')[0],
    errors='coerce',
)
combined['scenario_id'] = pd.to_numeric(
    combined['scenario'].str.extract(r'(\d+)')[0],
    errors='coerce',
).astype('Int64')

combined = combined.sort_values(
    ['subject_num', 'scenario_id', 'task', 'channel', 'subband']
).drop(columns=['subject_num']).reset_index(drop=True)

duplicate_key_rows = combined.duplicated(
    subset=['subject_id', 'scenario', 'task', 'channel', 'subband']
).sum()
null_summary = (combined.isna().mean() * 100).sort_values(ascending=False)
coverage = combined.groupby(['channel', 'subband']).size().reset_index(name='rows')

print(f'Final rows: {len(combined):,}')
print(f'Unique subjects: {combined["subject_id"].nunique():,}')
print(f'Duplicate key rows: {duplicate_key_rows:,}')
print('\nTop null percentages (%):')
display(null_summary.head(12))
print('\nChannel/Subband coverage:')
display(coverage)

combined.to_csv(OUTPUT_CSV, index=False)
print(f'Combined CSV saved to: {OUTPUT_CSV}')
display(combined.head(10))

Final rows: 27,531
Unique subjects: 170
Duplicate key rows: 0

Top null percentages (%):


erd_ers_pct                50.081726
task_power                 50.081726
baseline_power             50.081726
chain_ratio_mav            11.768552
chain_sequence_variance    11.768552
chain_sequence_mav         11.768552
chain_ratio_variance       11.768552
variance                    0.000000
mav                         0.000000
subband                     0.000000
channel                     0.000000
task                        0.000000
dtype: float64


Channel/Subband coverage:


,channel,subband,rows
0,C3,High_Beta,3059
1,C3,Low_Beta,3059
2,C3,Mu,3059
3,C4,High_Beta,3059
4,C4,Low_Beta,3059
5,C4,Mu,3059
6,CZ,High_Beta,3059
7,CZ,Low_Beta,3059
8,CZ,Mu,3059


Combined CSV saved to: /home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_ALS/WEB/features.csv/normal_features_combined.csv


,subject_id,scenario,task,channel,subband,mav,variance,baseline_power,task_power,erd_ers_pct,chain_ratio_mav,chain_ratio_variance,chain_sequence_mav,chain_sequence_variance,scenario_id
0,id1,scenario1,Resting,C3,High_Beta,5.082302,0.000089,NaN,NaN,NaN,0.5488,0.5305,1001010110000111011000111001101000101001010110...,1001010110000111011000111001101000101001010110...,1
1,id1,scenario1,Resting,C3,Low_Beta,3.395974,0.000034,NaN,NaN,NaN,0.5610,0.5610,1001110101010111111010110101101100101101011110...,1001110111010111111010110001101100101101011110...,1
2,id1,scenario1,Resting,C3,Mu,3.931533,0.000055,NaN,NaN,NaN,0.5244,0.5183,1101000101011011101100010011010011001101100101...,1101000101011011101100010011010011001101100101...,1
3,id1,scenario1,Resting,C4,High_Beta,4.151432,0.000044,NaN,NaN,NaN,0.5671,0.5610,1010111101001101110011010111010011010010101000...,1010111101001101110011010111010011010010101000...,1
4,id1,scenario1,Resting,C4,Low_Beta,2.680949,0.000018,NaN,NaN,NaN,0.5305,0.5305,1011011101001101110001100011011011010000111000...,1011011101001101110001100011011011010000111000...,1
5,id1,scenario1,Resting,C4,Mu,3.282657,0.000026,NaN,NaN,NaN,0.5305,0.5427,1011001110100110111100111001101101101100011100...,1111001110100110111100111001101101101100011100...,1
6,id1,scenario1,Resting,CZ,High_Beta,6.597786,0.000128,NaN,NaN,NaN,0.5427,0.5488,1001011000000110110010011010101101101001011110...,1001011001010110110010011010101001101001011110...,1
7,id1,scenario1,Resting,CZ,Low_Beta,2.703203,0.000015,NaN,NaN,NaN,0.5671,0.5671,1110101010010110110010110101101001101100011110...,1110101010010110110010110101101001101100011110...,1
8,id1,scenario1,Resting,CZ,Mu,3.467531,0.000024,NaN,NaN,NaN,0.5183,0.5244,1000011001101101010100101010100011101001111100...,1000011001101101010100101010100011101001111100...,1
9,id1,scenario1,Thinking,C3,High_Beta,13.166646,0.000331,0.064689,0.025508,-605685.7924,0.5119,0.5000,1001010001100010010111010010110000101111010111...,1001010001100010010111010010010000101111110111...,1
